In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()
import pandas as pd
data = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')

In [ ]:
# Task 2: Write your code here:Inspect the first few rows using head()
data.head()

In [ ]:
# Task 3: Write your code here:Display dataset information using info()
data.info()

In [ ]:
# Task 4: Write your code here:Show statistical description using describe()
data.describe()

In [ ]:
# Task 5: Write your code here:Plot the target distribution (delivery_time)
# Delivery_Time distribution (target variable)
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist(data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:Drop the 'Order_ID' column from the data
data.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:Handle missing values appropriately
print("Missing values remaining:", data.isnull().sum()) #checking for missing data

In [ ]:
#fill missing data
#fill categorical with mode
data['Weather'] = data['Weather'].fillna(data['Weather'].mode()[0])
data['Traffic_Level'] = data['Traffic_Level'].fillna(data['Traffic_Level'].mode()[0])
data['Time_of_Day'] = data['Time_of_Day'].fillna(data['Time_of_Day'].mode()[0])
#fill neumerical data with the mean
data['Delivery_Time'] = data['Delivery_Time'].fillna(data['Delivery_Time'].mean())
data['Courier_Experience_yrs'] = data['Courier_Experience_yrs'].fillna(data['Courier_Experience_yrs'].mean())
data.isnull().sum()


In [ ]:
# Task 3: Write your code here:Check and remove duplicates
duplicates = data.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
     print("Dropping Duplicates...")
     data.drop_duplicates(inplace=True)
     print("Duplicates Dropped.")
else:
     print("No Duplicate Samples Found.")
print(data.duplicated().sum())

In [ ]:
# Task 4: Write your code here:Encode categorical variables
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

categorical_cols = data.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
for col in categorical_cols:
    onehot = OneHotEncoder()
    data[col] = onehot.fit_transform(data[col].astype(str))
data.head()

In [ ]:
# Task 5: Write your code here:Apply feature scaling for all features
from sklearn.preprocessing import StandardScaler
features = data.columns.drop("Delivery_Time")  # dropping the target , we can not scale it

scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data.head()


In [ ]:
# Task 6: Write your code here:Check for target imbalance and state if it is imbalanced or not
# Delivery_Time distribution (it is imbalanced)
Delivery_Time_counts = data['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(Delivery_Time_counts.index, Delivery_Time_counts.values, color='teal')
plt.title('Delivery_Time  Distribution')
plt.xlabel('Delivery_Times Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Task 1: Write your code here:Split the dataset into features (X) and target (y)
X = data[features]
y = data['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True         # representative splits

)



In [ ]:
# Task 2,3,4,5: Write your code here:
#Use the correct split: KFold OR StratifiedKFold
from sklearn.model_selection import StratifiedKFold

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

#Train a RandomForest mode
#Evaluate using MAE (Mean Absolute Error) ONLY
#Print the averaged score across all folds

In [ ]:
#Evaluate using MAE (Mean Absolute Error) ONLY
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
y_pred = model.predict(X_test)

# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("MSE :", mse)
print("MAE :", mae)
print("RMSE:", rmse)

In [ ]:
#Print the averaged score across all folds

In [ ]:
# Task 1: Write your code here:Plot feature importance from your trained mode
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:Plot predicted delivery time histogram
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, y_pred, alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: